In [ ]:
import pandas as pd

response_df = pd.read_json(
    "Dataset/response.jsonl",
    lines=True
)

source_df = pd.read_json(
    "Dataset/source_info.jsonl",
    lines=True
)

print(response_df.shape)
print(source_df.shape)

FileNotFoundError: File RAGGuard/Dataset/response.jsonl does not exist

In [ ]:
combined_df = response_df.merge(
    source_df,
    on="source_id",
    how="left"
)

print(combined_df.shape)

(17790, 12)


In [ ]:
combined_df["target"] = combined_df["labels"].apply(
    lambda x: 1 if len(x) > 0 else 0
)

In [ ]:
model_df = combined_df[
    combined_df["quality"] == "good"
].copy()

print(model_df.shape)

(17617, 13)


In [ ]:
import re

def clean_text(text):
    text = str(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [ ]:
model_df["source_info"] = model_df["source_info"].apply(clean_text)
model_df["response"] = model_df["response"].apply(clean_text)

In [ ]:
print(
    model_df[
        ["source_info", "response", "target", "split"]
    ].head()
)

                                         source_info  \
0  Seventy years ago, Anne Frank died of typhus i...   
1  Seventy years ago, Anne Frank died of typhus i...   
2  Seventy years ago, Anne Frank died of typhus i...   
3  Seventy years ago, Anne Frank died of typhus i...   
4  Seventy years ago, Anne Frank died of typhus i...   

                                            response  target  split  
0  The Anne Frank House has revealed that Anne Fr...       0  train  
1  New research released by the Anne Frank House ...       0  train  
2  New research conducted by the Anne Frank House...       1  train  
3  Here is a summary of the article in 115 words ...       1  train  
4  Seventy years ago, Anne Frank died of typhus i...       0  train  


In [ ]:
print(model_df["target"].value_counts())

target
0    9976
1    7641
Name: count, dtype: int64


In [ ]:
print(model_df["split"].value_counts())

split
train    14942
test      2675
Name: count, dtype: int64


In [ ]:
print(
    pd.crosstab(
        model_df["split"],
        model_df["target"]
    )
)

target     0     1
split             
test    1732   943
train   8244  6698


In [ ]:
for i in range(3):
    row = model_df.iloc[i]

    print("=" * 80)
    print("SOURCE:")
    print(row["source_info"])

    print("\nRESPONSE:")
    print(row["response"])

    print("\nTARGET:")
    print(row["target"])

SOURCE:
Seventy years ago, Anne Frank died of typhus in a Nazi concentration camp at the age of 15. Just two weeks after her supposed death on March 31, 1945, the Bergen-Belsen concentration camp where she had been imprisoned was liberated -- timing that showed how close the Jewish diarist had been to surviving the Holocaust. But new research released by the Anne Frank House shows that Anne and her older sister, Margot Frank, died at least a month earlier than previously thought. Researchers re-examined archives of the Red Cross, the International Training Service and the Bergen-Belsen Memorial, along with testimonies of survivors. They concluded that Anne and Margot probably did not survive to March 1945 -- contradicting the date of death which had previously been determined by Dutch authorities. In 1944, Anne and seven others hiding in the Amsterdam secret annex were arrested and sent to the Auschwitz-Birkenau concentration camp. Anne Frank's final entry. That same year, Anne and Mar

In [ ]:
print("Shape:", model_df.shape)

print("\nTarget distribution:")
print(model_df["target"].value_counts())

print("\nSplit distribution:")
print(model_df["split"].value_counts())

print("\nTarget by split:")
print(pd.crosstab(model_df["split"], model_df["target"]))

Shape: (17617, 13)

Target distribution:
target
0    9976
1    7641
Name: count, dtype: int64

Split distribution:
split
train    14942
test      2675
Name: count, dtype: int64

Target by split:
target     0     1
split             
test    1732   943
train   8244  6698


In [ ]:
from transformers import AutoTokenizer

model_name = "microsoft/deberta-v3-base"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_fast=False
)

print("Tokenizer loaded successfully!")

Tokenizer loaded successfully!


In [ ]:
import torch
import transformers
import sentencepiece

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())
print("SentencePiece:", sentencepiece.__version__)

PyTorch: 2.14.0+cpu
Transformers: 4.57.1
CUDA available: False
SentencePiece: 0.2.2


In [ ]:
sample = model_df.iloc[0]

print("SOURCE:")
print(sample["source_info"])

print("\n" + "="*80 + "\n")

print("RESPONSE:")
print(sample["response"])

print("\nTARGET:", sample["target"])

SOURCE:
Seventy years ago, Anne Frank died of typhus in a Nazi concentration camp at the age of 15. Just two weeks after her supposed death on March 31, 1945, the Bergen-Belsen concentration camp where she had been imprisoned was liberated -- timing that showed how close the Jewish diarist had been to surviving the Holocaust. But new research released by the Anne Frank House shows that Anne and her older sister, Margot Frank, died at least a month earlier than previously thought. Researchers re-examined archives of the Red Cross, the International Training Service and the Bergen-Belsen Memorial, along with testimonies of survivors. They concluded that Anne and Margot probably did not survive to March 1945 -- contradicting the date of death which had previously been determined by Dutch authorities. In 1944, Anne and seven others hiding in the Amsterdam secret annex were arrested and sent to the Auschwitz-Birkenau concentration camp. Anne Frank's final entry. That same year, Anne and Mar

In [ ]:
source = sample["source_info"]
response = sample["response"]

encoded = tokenizer(
    source,
    response,
    truncation=True,
    max_length=512
)

print(encoded.keys())
print("Number of tokens:", len(encoded["input_ids"]))

Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.


KeysView({'input_ids': [1, 42236, 388, 824, 261, 6477, 4279, 1675, 265, 113952, 267, 266, 10794, 5219, 2964, 288, 262, 905, 265, 706, 260, 1063, 375, 1033, 385, 342, 2902, 1142, 277, 1014, 1947, 261, 11368, 261, 262, 25500, 271, 983, 66618, 5219, 2964, 399, 373, 330, 331, 19143, 284, 30264, 341, 271, 5703, 272, 1938, 361, 828, 262, 3558, 3752, 65988, 330, 331, 264, 11309, 262, 14856, 260, 420, 353, 677, 1315, 293, 262, 6477, 4279, 1097, 1057, 272, 6477, 263, 342, 1921, 2649, 261, 43203, 4279, 261, 1675, 288, 668, 266, 788, 1532, 354, 2154, 708, 260, 12426, 961, 271, 54041, 11263, 265, 262, 1813, 4241, 261, 262, 1282, 4202, 1562, 263, 262, 25500, 271, 983, 66618, 5450, 261, 641, 275, 34146, 265, 9617, 260, 450, 5988, 272, 6477, 263, 43203, 873, 464, 298, 4601, 264, 1014, 11368, 341, 271, 62545, 262, 1043, 265, 1142, 319, 330, 2154, 331, 2662, 293, 5862, 3081, 260, 344, 13606, 261, 6477, 263, 1720, 690, 7864, 267, 262, 10150, 3002, 35263, 332, 3740, 263, 1253, 264, 262, 40257, 271, 11752

In [ ]:
source_tokens = tokenizer(
    source,
    truncation=False
)["input_ids"]

response_tokens = tokenizer(
    response,
    truncation=False
)["input_ids"]

print("Source tokens:", len(source_tokens))
print("Response tokens:", len(response_tokens))
print("Total separately:", len(source_tokens) + len(response_tokens))

Source tokens: 569
Response tokens: 133
Total separately: 702


In [ ]:
from tqdm import tqdm

source_lengths = []
response_lengths = []

for _, row in tqdm(model_df.iterrows(), total=len(model_df)):
    source_len = len(
        tokenizer(
            row["source_info"],
            truncation=False
        )["input_ids"]
    )

    response_len = len(
        tokenizer(
            row["response"],
            truncation=False
        )["input_ids"]
    )

    source_lengths.append(source_len)
    response_lengths.append(response_len)

model_df["source_tokens"] = source_lengths
model_df["response_tokens"] = response_lengths
model_df["total_tokens"] = (
    model_df["source_tokens"] +
    model_df["response_tokens"]
)

print("Total examples:", len(model_df))
print(
    "Examples <= 512:",
    (model_df["total_tokens"] <= 512).sum()
)
print(
    "Examples > 512:",
    (model_df["total_tokens"] > 512).sum()
)

100%|██████████| 17617/17617 [00:42<00:00, 416.39it/s]

Total examples: 17617
Examples <= 512: 4889
Examples > 512: 12728


In [ ]:
print("Response token statistics:")
print(model_df["response_tokens"].describe())

print("\nSource token statistics:")
print(model_df["source_tokens"].describe())

Response token statistics:
count    17617.00000
mean       160.49526
std         67.51949
min          3.00000
25%        112.00000
50%        153.00000
75%        199.00000
max        772.00000
Name: response_tokens, dtype: float64

Source token statistics:
count    17617.000000
mean       597.471476
std        320.518839
min        143.000000
25%        370.000000
50%        530.000000
75%        759.000000
max       2189.000000
Name: source_tokens, dtype: float64


In [ ]:
for limit in [256, 384, 512]:
    count = (model_df["response_tokens"] <= limit).sum()
    percentage = count / len(model_df) * 100
    
    print(
        f"Response <= {limit} tokens: "
        f"{count} ({percentage:.2f}%)"
    )

Response <= 256 tokens: 15865 (90.06%)
Response <= 384 tokens: 17576 (99.77%)
Response <= 512 tokens: 17616 (99.99%)


In [ ]:
long_response = model_df[
    model_df["response_tokens"] > 512
]

print(long_response[
    ["response_tokens", "source_tokens", "target"]
])

      response_tokens  source_tokens  target
2280              772            805       0


In [ ]:
model_df = model_df[
    model_df["response_tokens"] <= 512
].copy()

print("Transformer dataset size:", len(model_df))
print("Train:", (model_df["split"] == "train").sum())
print("Test:", (model_df["split"] == "test").sum())

Transformer dataset size: 17616
Train: 14941
Test: 2675


In [ ]:
from transformers import AutoTokenizer

model_name = "answerdotai/ModernBERT-base"

modern_tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded successfully!")
print("Maximum length:", modern_tokenizer.model_max_length)

Tokenizer loaded successfully!
Maximum length: 8192


In [ ]:
sample = model_df.iloc[0]

encoded = modern_tokenizer(
    sample["source_info"],
    sample["response"],
    truncation=False
)

print("Total tokens:", len(encoded["input_ids"]))
print("Model maximum:", modern_tokenizer.model_max_length)

Total tokens: 737
Model maximum: 8192


In [ ]:
longest_sample = model_df.loc[
    model_df["total_tokens"].idxmax()
]

encoded_longest = modern_tokenizer(
    longest_sample["source_info"],
    longest_sample["response"],
    truncation=False
)

print("Longest example tokens:", len(encoded_longest["input_ids"]))
print("Model maximum:", modern_tokenizer.model_max_length)

Longest example tokens: 2609
Model maximum: 8192


In [ ]:
from transformers import AutoModelForSequenceClassification

modern_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

print("Model loaded successfully!")
print("Number of labels:", modern_model.config.num_labels)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded successfully!
Number of labels: 2


In [ ]:
import torch

inputs = modern_tokenizer(
    sample["source_info"],
    sample["response"],
    return_tensors="pt",
    truncation=False
)

with torch.no_grad():
    outputs = modern_model(**inputs)

print("Logits:", outputs.logits)
print("Logit shape:", outputs.logits.shape)

Logits: tensor([[-0.4559, -0.0335]])
Logit shape: torch.Size([1, 2])


In [ ]:
## Final ModernBERT Training – Full Dataset

In [ ]:

import os
import pandas as pd
from sklearn.model_selection import train_test_split

# Create separate folders for final model and results
os.makedirs("../Models/modernbert/final", exist_ok=True)
os.makedirs("../Results", exist_ok=True)

# Separate the original training and testing splits
train_pool = model_df[model_df["split"] == "train"].copy()
test_df = model_df[model_df["split"] == "test"].copy()

# Create validation set from training data
train_df, val_df = train_test_split(
    train_pool,
    test_size=0.10,
    random_state=42,
    stratify=train_pool["target"]
)

# Reset indexes
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("FULL DATASET READY")
print("Training samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Testing samples:", len(test_df))

print("\nTraining label distribution:")
print(train_df["target"].value_counts())

print("\nValidation label distribution:")
print(val_df["target"].value_counts())

print("\nTesting label distribution:")
print(test_df["target"].value_counts())


NameError: name 'model_df' is not defined